# Ch 11 · Equations of Motion, Kinematics and the Mass Matrix

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/master/dpdk-research/mujoco-warp-course/notebooks/11-equations-of-motion-kinematics-mass-matrix.ipynb)

Companion notebook for the chapter page `chapters/11-equations-of-motion-kinematics-mass-matrix.html` ([page](../chapters/11-equations-of-motion-kinematics-mass-matrix.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
# (not a rendering notebook: MUJOCO_GL left unset)
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

## How to use this notebook
Run the setup cell first. Every example below is the same file as on the chapter page (`examples/11-equations-of-motion-kinematics-mass-matrix/`). Budgets switch on `ON_GPU`: small on CPU, large on a Colab GPU. Timings you see are your own device's numbers.

## 1-2. The equation of motion and the mass matrix
`M(q) qacc + c(q, qdot) = tau + J^T f`. Example 11.1 derives `M` and `c` of the double pendulum by hand (Lagrange) and compares with MuJoCo C; Example 11.2 checks that `0.5 qvel^T M qvel` is the kinetic energy in every world.

In [ ]:
# Example 11.1 - ex1_two_link_numpy.py
"""Example 11.1 - derive M(q) and c(q, qdot) of the double pendulum by hand (Lagrange) in NumPy.

The PENDULUM model (shared model M2) is a planar 2-link arm: two hinges about +y, two capsules of
mass 1 and length 0.5. From the Lagrangian L = T - V we get the manipulator equation
    M(q) qacc + c(q, qdot) = tau
with c = Coriolis/centrifugal terms + gravity. We build M and c from the link masses, com offsets
and inertias that MuJoCo compiled, then compare with mujoco.mj_fullM and mjData.qfrc_bias at three
random states. This example uses only MuJoCo C (float64), so the errors are at round-off level.
Run: python ex1_two_link_numpy.py
"""
import mujoco
import numpy as np

np.set_printoptions(precision=4, suppress=True)

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

mjm = mujoco.MjModel.from_xml_string(PENDULUM)
mjd = mujoco.MjData(mjm)

# --- link parameters, read from the compiled model --------------------------------------
m1, m2 = mjm.body_mass[1], mjm.body_mass[2]
lc1, lc2 = -mjm.body_ipos[1, 2], -mjm.body_ipos[2, 2]     # com distance from each hinge
l1 = -mjm.body_pos[2, 2]                                     # hinge1 -> hinge2 distance


def inertia_about_y(b):
    """Moment of inertia of body b about the y axis through its com."""
    R = np.zeros(9)
    mujoco.mju_quat2Mat(R, mjm.body_iquat[b])
    R = R.reshape(3, 3)
    return (R @ np.diag(mjm.body_inertia[b]) @ R.T)[1, 1]


I1, I2 = inertia_about_y(1), inertia_about_y(2)
g = -mjm.opt.gravity[2]
print(f"m = ({m1:.3f}, {m2:.3f})  lc = ({lc1:.3f}, {lc2:.3f})  l1 = {l1:.3f}  "
      f"I_yy = ({I1:.6f}, {I2:.6f})")


def mass_matrix(q):
    """M(q) from T = 0.5 qdot^T M qdot (planar 2-link, hinges about the same axis)."""
    c2 = np.cos(q[1])
    a = I1 + I2 + m1 * lc1**2 + m2 * (l1**2 + lc2**2 + 2 * l1 * lc2 * c2)
    b = I2 + m2 * (lc2**2 + l1 * lc2 * c2)
    d = I2 + m2 * lc2**2
    return np.array([[a, b], [b, d]])


def bias(q, qd):
    """c(q, qdot) = Christoffel terms (Coriolis + centrifugal) + dV/dq (gravity)."""
    s2 = np.sin(q[1])
    # dM/dq2 (M does not depend on q1)
    dM = np.array([[-2 * m2 * l1 * lc2 * s2, -m2 * l1 * lc2 * s2],
                   [-m2 * l1 * lc2 * s2, 0.0]])
    dMdq = [np.zeros((2, 2)), dM]
    cor = np.zeros(2)
    for i in range(2):
        for j in range(2):
            for k in range(2):
                cor[i] += (dMdq[k][i, j] - 0.5 * dMdq[i][j, k]) * qd[j] * qd[k]
    # V = sum m g z with z1 = -lc1 cos q1, z2 = -l1 cos q1 - lc2 cos(q1 + q2) (+ const)
    grav = np.array([m1 * g * lc1 * np.sin(q[0]) + m2 * g * (l1 * np.sin(q[0])
                                                              + lc2 * np.sin(q[0] + q[1])),
                     m2 * g * lc2 * np.sin(q[0] + q[1])])
    return cor + grav


rng = np.random.default_rng(0)
full = np.zeros((mjm.nv, mjm.nv))
err_M, err_c = 0.0, 0.0
for trial in range(3):
    mjd.qpos[:] = rng.uniform(-np.pi, np.pi, 2)
    mjd.qvel[:] = rng.uniform(-3, 3, 2)
    mujoco.mj_forward(mjm, mjd)
    mujoco.mj_fullM(mjm, mjd, full)   # MuJoCo 4 signature: (m, d, dst)
    M_hand = mass_matrix(mjd.qpos)
    c_hand = bias(mjd.qpos, mjd.qvel)
    err_M = max(err_M, np.abs(M_hand - full).max())
    err_c = max(err_c, np.abs(c_hand - mjd.qfrc_bias).max())
    print(f"state {trial}: q = {mjd.qpos}, qdot = {mjd.qvel}")
    print("  M (hand)      =", M_hand.ravel(), "\n  M (mj_fullM)  =", full.ravel())
    print("  c (hand)      =", c_hand, "\n  c (qfrc_bias) =", mjd.qfrc_bias)
print(f"max |M_hand - mj_fullM| = {err_M:.2e}")
print(f"max |c_hand - qfrc_bias| = {err_c:.2e}")

In [ ]:
# Example 11.2 - ex2_kinetic_energy.py
"""Example 11.2 - the mass matrix is the kinetic energy: T = 0.5 qvel^T M qvel, per world.

PENDULUM (energy flag enabled) in NWORLD worlds with random qpos/qvel. After mjw.forward we
1. densify each world's CSR mass matrix d.M and evaluate 0.5 qvel^T M qvel in NumPy,
2. read MJWarp's own kinetic energy d.energy[:, 1] (sensor.energy_vel),
3. sum 0.5 m |v_com|^2 + 0.5 w^T I w over the bodies in MuJoCo C (mj_objectVelocity),
and print all three per world. They agree to float32 round-off.
Run: python ex2_kinetic_energy.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8


def csr_to_dense(mjm, values):
    """Symmetric dense M from MJWarp's CSR lower triangle (one world)."""
    M = np.zeros((mjm.nv, mjm.nv))
    for i in range(mjm.nv):
        for k in range(mjm.M_rownnz[i]):
            adr = mjm.M_rowadr[i] + k
            j = mjm.M_colind[adr]
            M[i, j] = M[j, i] = values[adr]
    return M


mjm = mujoco.MjModel.from_xml_string(PENDULUM)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)

rng = np.random.default_rng(1)
qpos = rng.uniform(-np.pi, np.pi, (NWORLD, mjm.nq)).astype(np.float32)
qvel = rng.uniform(-4, 4, (NWORLD, mjm.nv)).astype(np.float32)
d.qpos.assign(qpos)
d.qvel.assign(qvel)
mjw.forward(m, d)

M_csr = d.M.numpy()               # (nworld, nC)
energy = d.energy.numpy()         # (nworld, 2): [potential, kinetic]

mjd = mujoco.MjData(mjm)
vel = np.zeros(6)
print("world  0.5 v^T M v   d.energy[:,1]   sum over bodies (C)")
worst = 0.0
for w in range(min(NWORLD, 8)):
    M = csr_to_dense(mjm, M_csr[w])
    t_m = 0.5 * qvel[w] @ M @ qvel[w]
    mjd.qpos[:], mjd.qvel[:] = qpos[w], qvel[w]
    mujoco.mj_forward(mjm, mjd)
    t_b = 0.0
    for b in range(1, mjm.nbody):
        # velocity of the body's inertial frame, in that frame: (angular, linear)
        mujoco.mj_objectVelocity(mjm, mjd, mujoco.mjtObj.mjOBJ_BODY, b, vel, 1)
        w_loc, v_loc = vel[:3], vel[3:]
        t_b += 0.5 * mjm.body_mass[b] * v_loc @ v_loc + 0.5 * w_loc @ (mjm.body_inertia[b] * w_loc)
    worst = max(worst, abs(t_m - energy[w, 1]), abs(t_m - t_b))
    print(f"{w:5d}  {t_m:11.5f}   {energy[w, 1]:13.5f}   {t_b:12.5f}")
print(f"max abs difference: {worst:.2e} (float32 MJWarp vs float64 C)")

## 3. Spatial vectors and the com frame
`cvel` is (angular, linear) and its linear part is the velocity of the point at the subtree com of the root.

In [ ]:
# Example 11.3 - ex3_spatial_vectors.py
"""Example 11.3 - spatial velocity cvel: 6 numbers (angular, linear) in the com frame.

d.cvel[w, b] is a wp.spatial_vector: first 3 = angular velocity, last 3 = linear velocity of the
point that coincides with c = subtree_com[root of b] (the "com frame"), all in world axes.
For any point p moving with body b:  v_p = v_lin + w x (p - c).
We check (1) link1, which turns about a fixed hinge anchor a: v_lin = w x (c - a), and (2) the
velocity of each body com xipos against a finite difference of MuJoCo C kinematics.
Run: python ex3_spatial_vectors.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""
mjm = mujoco.MjModel.from_xml_string(PENDULUM)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1)
q0 = np.array([0.6, -0.9])
qd0 = np.array([1.5, 2.0])
d.qpos.assign(q0[None].astype(np.float32))
d.qvel.assign(qd0[None].astype(np.float32))
mjw.forward(m, d)

cvel = d.cvel.numpy()[0]            # (nbody, 6)
c = d.subtree_com.numpy()[0, 1]     # com of the tree rooted at link1 (the "com frame" origin)
xanchor = d.xanchor.numpy()[0]      # (njnt, 3)
xipos = d.xipos.numpy()[0]
print("cvel shape per world:", d.cvel.numpy().shape[1:], "dtype", d.cvel.dtype.__name__)
for b in (1, 2):
    print(f"body {b} ({mjm.body(b).name}): angular = {cvel[b, :3]}, linear = {cvel[b, 3:]}")
print("com frame origin c = subtree_com[1] =", c)

# (1) link1 rotates about its hinge anchor: the linear part is w x (c - anchor)
w1, v1 = cvel[1, :3], cvel[1, 3:]
res1 = np.abs(v1 - np.cross(w1, c - xanchor[0])).max()
print(f"link1: |v_lin - w x (c - xanchor)| = {res1:.2e}")

# (2) com velocity of each body: v_lin + w x (xipos - c) vs finite difference in MuJoCo C
mjd = mujoco.MjData(mjm)
eps = 1e-6
pos = []
for s in (-1, 1):
    mjd.qpos[:] = q0 + s * eps * qd0
    mujoco.mj_kinematics(mjm, mjd)
    pos.append(mjd.xipos.copy())
v_fd = (pos[1] - pos[0]) / (2 * eps)
for b in (1, 2):
    v_pred = cvel[b, 3:] + np.cross(cvel[b, :3], xipos[b] - c)
    print(f"body {b}: v_com from cvel = {v_pred}, finite diff = {v_fd[b]}, "
          f"residual = {np.abs(v_pred - v_fd[b]).max():.2e}")

## 4-5. Joint math, the kinematics outputs, and the branch walk
`smooth.kinematics` launches `_kinematics_branch` with `dim=(nworld, nbranch)`, one thread per (world, branch) walking root to leaf, then per-body, per-geom and per-site poses.

In [ ]:
# Example 11.4 - ex4_branches.py
"""Example 11.4 - the humanoid body tree as branches (one kinematics thread per branch).

put_model splits the body tree into root-to-leaf branches (io.py: one branch per leaf body).
smooth.py::kinematics launches _kinematics_branch with dim=(nworld, m.nbranch); each thread walks
body_branches[body_branch_start[b]:body_branch_start[b + 1]] from root to leaf. Shared prefixes
(the torso) are recomputed in every branch. We also print body_subtreenum, the length of each
body's contiguous subtree range, and the level-by-level alternative (m.body_tree).
Run: python ex4_branches.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

HUMANOID_PATH = epath.resource_path("mujoco_warp") / "test_data/humanoid/humanoid.xml"
mjm = mujoco.MjModel.from_xml_path(str(HUMANOID_PATH))
m = mjw.put_model(mjm)

names = [mjm.body(i).name for i in range(mjm.nbody)]
branches_flat = m.body_branches.numpy()
start = m.body_branch_start.numpy()
subtreenum = m.body_subtreenum.numpy()
branches = [branches_flat[start[b]:start[b + 1]].tolist() for b in range(m.nbranch)]

print(f"nbody = {mjm.nbody}, nv = {mjm.nv}, nbranch = {m.nbranch}")
print("body_branch_start =", start.tolist())
for b, br in enumerate(branches):
    print(f"  branch {b}: {br}  ({' > '.join(names[i] for i in br)})")
work = sum(len(br) for br in branches)
print(f"body visits by the branch walk: {work} for {mjm.nbody - 1} bodies "
      f"({work - (mjm.nbody - 1)} redundant, the torso alone {sum(1 in br for br in branches)}x)")
print("body_subtreenum =", subtreenum.tolist())
print("subtree of body 3 (waist_lower) = range", (3, 3 + int(subtreenum[3])),
      "->", [names[i] for i in range(3, 3 + int(subtreenum[3]))])
levels = [lvl.numpy().tolist() for lvl in m.body_tree]
print(f"level-by-level alternative: {len(levels)} levels (one launch + barrier each):", levels)

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"names": names, "parent": mjm.body_parentid.tolist(), "branches": branches,
                   "subtreenum": subtreenum.tolist(), "levels": levels,
                   "mass": [round(float(x), 3) for x in mjm.body_mass]}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

In [ ]:
# Example 11.5 - ex5_kinematics_only.py
"""Example 11.5 - run only the kinematics stage (mjw.kinematics) and compare with MuJoCo C.

HUMANOID in NWORLD worlds, each with its own random qpos (free-joint quaternion normalized,
hinges uniform in [-1, 1] rad). mjw.kinematics writes xpos/xquat/xmat, xipos/ximat, xanchor/xaxis,
geom_xpos/geom_xmat and site_xpos/site_xmat (the humanoid has no sites) for all worlds in five
launches. We compare every world with mujoco.mj_kinematics on the same qpos.
Run: python ex5_kinematics_only.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath

np.set_printoptions(precision=4, suppress=True)

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NCHECK = min(NWORLD, 64)      # worlds compared with MuJoCo C (the C loop is serial)

HUMANOID_PATH = epath.resource_path("mujoco_warp") / "test_data/humanoid/humanoid.xml"
mjm = mujoco.MjModel.from_xml_path(str(HUMANOID_PATH))
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)

rng = np.random.default_rng(2)
qpos = np.tile(mjm.qpos0, (NWORLD, 1))
qpos[:, 0:3] += rng.normal(0, 0.3, (NWORLD, 3))              # free joint position
quat = rng.normal(size=(NWORLD, 4))
qpos[:, 3:7] = quat / np.linalg.norm(quat, axis=1, keepdims=True)   # free joint orientation
qpos[:, 7:] = rng.uniform(-1, 1, (NWORLD, mjm.nq - 7))        # 21 hinges
d.qpos.assign(qpos.astype(np.float32))

mjw.kinematics(m, d)                                         # just the kinematics stage

fields = ["xpos", "xquat", "xipos", "xanchor", "xaxis", "geom_xpos", "geom_xmat"]
got = {f: getattr(d, f).numpy() for f in fields}
err = {f: 0.0 for f in fields}
mjd = mujoco.MjData(mjm)
for w in range(NCHECK):
    mjd.qpos[:] = qpos[w]
    mujoco.mj_kinematics(mjm, mjd)
    for f in fields:
        ref = getattr(mjd, f).reshape(got[f][w].shape)   # C stores xmat as 9 numbers
        err[f] = max(err[f], np.abs(got[f][w] - ref).max())
print(f"jnt types: {sorted(set(mjm.jnt_type.tolist()))} (0=free, 3=hinge), nworld = {NWORLD}")
print("xpos of the torso in the first 3 worlds:\n", got["xpos"][:3, 1])
for f in fields:
    print(f"max |{f:9s} - C| over {NCHECK} worlds: {err[f]:.2e}   shape {got[f].shape}")

### Your own kinematics kernel
A closed-form planar FK kernel, one thread per world. Timings are your device's numbers (on CPU they are CPU numbers, not GPU performance).

In [ ]:
# Example 11.11 - ex11_my_fk_kernel.py
"""Example 11.11 - write your own forward-kinematics kernel for the planar 2-link arm.

One Warp thread per world computes the tip position of the PENDULUM in closed form:
    tip = base + R_y(q1) (0, 0, -l1) + R_y(q1 + q2) (0, 0, -l2),
where a rotation by angle a about +y maps (0, 0, -l) to (-l sin a, 0, -l cos a).
We compare with d.site_xpos written by mjw.kinematics and time both. The general kernel walks
branches, joints and quaternions for any tree; ours knows the model, so it does less work.
Timings are from whatever device you run on (on this page: a CPU dev box, not a GPU).
Run: python ex11_my_fk_kernel.py
"""
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""
ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8
NREP = 200 if ON_GPU else 20


@wp.kernel
def planar_fk(qpos: wp.array2d[float], base: wp.vec3, l1: float, l2: float,
              tip_out: wp.array[wp.vec3]):
    worldid = wp.tid()                       # one thread per world
    a1 = qpos[worldid, 0]
    a12 = a1 + qpos[worldid, 1]
    p = base + wp.vec3(-l1 * wp.sin(a1), 0.0, -l1 * wp.cos(a1))
    tip_out[worldid] = p + wp.vec3(-l2 * wp.sin(a12), 0.0, -l2 * wp.cos(a12))


mjm = mujoco.MjModel.from_xml_string(PENDULUM)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
rng = np.random.default_rng(5)
d.qpos.assign(rng.uniform(-np.pi, np.pi, (NWORLD, mjm.nq)).astype(np.float32))

base = wp.vec3(*mjm.body_pos[1])             # hinge1 anchor (0, 0, 1.5)
l1, l2 = float(-mjm.body_pos[2, 2]), float(-mjm.site_pos[0, 2])
tip = wp.zeros(NWORLD, dtype=wp.vec3)

wp.launch(planar_fk, dim=NWORLD, inputs=[d.qpos, base, l1, l2], outputs=[tip])
mjw.kinematics(m, d)                         # warm up (JIT) both, then compare
err = np.abs(tip.numpy() - d.site_xpos.numpy()[:, 0]).max()
print("tip, world 0 (ours)        :", tip.numpy()[0])
print("site_xpos, world 0 (mjw)   :", d.site_xpos.numpy()[0, 0])
print(f"max |ours - mjw.kinematics| over {NWORLD} worlds: {err:.2e}")


def timed(fn):
    wp.synchronize()
    t0 = time.perf_counter()
    for _ in range(NREP):
        fn()
    wp.synchronize()
    return (time.perf_counter() - t0) / NREP


t_ours = timed(lambda: wp.launch(planar_fk, dim=NWORLD, inputs=[d.qpos, base, l1, l2],
                                 outputs=[tip]))
t_mjw = timed(lambda: mjw.kinematics(m, d))
print(f"device {wp.get_device()}: our kernel {t_ours * 1e6:.1f} us/call (1 launch), "
      f"mjw.kinematics {t_mjw * 1e6:.1f} us/call (5 launches), nworld = {NWORLD}")

## 6. Subtrees as contiguous ranges

In [ ]:
# Example 11.6 - ex6_subtree_ranges.py
"""Example 11.6 - subtree centers of mass from contiguous body ranges.

MuJoCo numbers bodies so that every parent comes before its children (depth-first order). Then the
subtree of body b is exactly the index range [b, b + body_subtreenum[b]). smooth.py::_subtree_com
uses this: one thread per (world, body) sums mass * xipos over its range, no per-level launches.
The world body (0) is summed separately from the tree roots (_subtree_com_world).
We redo the sum in NumPy from d.xipos and compare with mjw.com_pos.
Run: python ex6_subtree_ranges.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath

np.set_printoptions(precision=4, suppress=True)

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8

HUMANOID_PATH = epath.resource_path("mujoco_warp") / "test_data/humanoid/humanoid.xml"
mjm = mujoco.MjModel.from_xml_path(str(HUMANOID_PATH))
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)

rng = np.random.default_rng(3)
qpos = np.tile(mjm.qpos0, (NWORLD, 1))
qpos[:, 7:] = rng.uniform(-1, 1, (NWORLD, mjm.nq - 7))
d.qpos.assign(qpos.astype(np.float32))
mjw.kinematics(m, d)
mjw.com_pos(m, d)               # subtree_com, cinert, cdof

subtreenum = m.body_subtreenum.numpy()
mass = mjm.body_mass
xipos = d.xipos.numpy()                     # (nworld, nbody, 3)
ours = np.zeros_like(xipos)
for b in range(1, mjm.nbody):
    rng_b = slice(b, b + subtreenum[b])     # the contiguous subtree range
    ours[:, b] = (mass[rng_b, None] * xipos[:, rng_b]).sum(1) / mass[rng_b].sum()
roots = [b for b in range(1, mjm.nbody) if mjm.body_parentid[b] == 0]
ours[:, 0] = sum(mjm.body_subtreemass[r] * ours[:, r] for r in roots) / mjm.body_subtreemass[0]

got = d.subtree_com.numpy()
print("body_subtreenum:", subtreenum.tolist())
print("tree roots (children of the world body):", roots)
print("subtree ranges:", {b: (b, int(b + subtreenum[b])) for b in (1, 3, 5, 11)})
print("world 0 subtree_com[1] (whole humanoid):", got[0, 1])
print("NumPy contiguous-range sum           :", ours[0, 1])
print(f"max |NumPy - mjw.com_pos| over {NWORLD} worlds x {mjm.nbody} bodies: "
      f"{np.abs(ours - got).max():.2e}")
print("body_subtreemass (Model constant, set at compile):", np.round(mjm.body_subtreemass[:6], 3))

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"names": [mjm.body(i).name for i in range(mjm.nbody)],
                   "parent": mjm.body_parentid.tolist(), "subtreenum": subtreenum.tolist(),
                   "mass": [round(float(x), 3) for x in mass],
                   "subtreemass": [round(float(x), 3) for x in mjm.body_subtreemass]},
                  f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 7. com_pos: cdof is the Jacobian

In [ ]:
# Example 11.7 - ex7_cdof_vs_jac.py
"""Example 11.7 - cdof is the Jacobian, one column per dof, in the com frame.

A planar 3-link hinge arm. For a hinge, smooth.py::_cdof writes
    cdof = (axis, axis x (c - xanchor))          c = subtree_com[root]
i.e. the spatial velocity (in the com frame) produced by qdot_j = 1. The Jacobian of a point p
on body b is then, for every dof j that is an ancestor of b:
    jacr[:, j] = cdof_ang,   jacp[:, j] = cdof_lin + cdof_ang x (p - c)
(support.py::jac_dof). We rebuild mj_jacBody's jacp/jacr for every body from cdof.
Run: python ex7_cdof_vs_jac.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ARM3 = """
<mujoco model="arm3">
  <worldbody>
    <body name="l1" pos="0 0 0">
      <joint name="j1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .4 0 0" size=".03" mass="1"/>
      <body name="l2" pos=".4 0 0">
        <joint name="j2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 .3 0 0" size=".03" mass=".7"/>
        <body name="l3" pos=".3 0 0">
          <joint name="j3" type="hinge" axis="0 1 0"/>
          <geom type="capsule" fromto="0 0 0 .2 0 0" size=".03" mass=".4"/>
          <site name="hand" pos=".2 0 0"/>
        </body>
      </body>
    </body>
  </worldbody>
</mujoco>
"""

mjm = mujoco.MjModel.from_xml_string(ARM3)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1)
q = np.array([0.5, -0.8, 0.9])
d.qpos.assign(q[None].astype(np.float32))
mjw.kinematics(m, d)
mjw.com_pos(m, d)

cdof = d.cdof.numpy()[0]                   # (nv, 6)
c = d.subtree_com.numpy()[0, 1]            # com frame origin (root of this tree: body 1)
xpos = d.xpos.numpy()[0]
print("com frame origin c =", c)
print("cdof rows (angular | linear):")
for j in range(mjm.nv):
    print(f"  dof {j}: {cdof[j, :3]} | {cdof[j, 3:]}")

mjd = mujoco.MjData(mjm)
mjd.qpos[:] = q
mujoco.mj_kinematics(mjm, mjd)
mujoco.mj_comPos(mjm, mjd)
jacp_c, jacr_c = np.zeros((3, mjm.nv)), np.zeros((3, mjm.nv))
worst = 0.0
for b in range(1, mjm.nbody):
    mujoco.mj_jacBody(mjm, mjd, jacp_c, jacr_c, b)
    jacp, jacr = np.zeros((3, mjm.nv)), np.zeros((3, mjm.nv))
    for j in range(mjm.nv):
        if mjm.dof_bodyid[j] <= b:          # in this chain: dof j is an ancestor of body b
            jacr[:, j] = cdof[j, :3]
            jacp[:, j] = cdof[j, 3:] + np.cross(cdof[j, :3], xpos[b] - c)
    worst = max(worst, np.abs(jacp - jacp_c).max(), np.abs(jacr - jacr_c).max())
    if b == 3:
        print("jacp of body l3 from cdof:\n", jacp, "\nmj_jacBody:\n", jacp_c)
print(f"max |jac from cdof - mj_jacBody| over all bodies: {worst:.2e}")

if args.dump_json:
    r4 = lambda a: np.round(np.asarray(a, dtype=float), 4).tolist()   # float64 for clean JSON
    out = {"xpos": r4(xpos), "xanchor": r4(d.xanchor.numpy()[0]),
           "site": r4(d.site_xpos.numpy()[0, 0]), "c": r4(c), "cdof": r4(cdof), "q": q.tolist()}
    with open(args.dump_json, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## 8. crb and the CSR mass matrix

In [ ]:
# Example 11.8 - ex8_csr_to_dense.py
"""Example 11.8 - read MJWarp's CSR mass matrix d.M and turn it into a dense matrix.

d.M has shape (nworld, nC). Row i of the lower triangle stores the entries M[i, j] for the dofs j
that are ancestors of dof i (and i itself, last): M_rowadr[i] is where the row starts, M_rownnz[i]
how many entries it has, M_colind[adr] the column of each entry (these three live on mjModel).
We densify the PENDULUM at qpos0 and compare with mujoco.mj_fullM, then print the sparsity of the
HUMANOID (nC = 243 of 27 x 27 = 729 entries).
Run: python ex8_csr_to_dense.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath

np.set_printoptions(precision=4, suppress=True)

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

def csr_to_dense(mjm, values):
    """Symmetric dense M from the CSR lower triangle of one world."""
    M = np.zeros((mjm.nv, mjm.nv))
    for i in range(mjm.nv):
        for k in range(mjm.M_rownnz[i]):
            adr = mjm.M_rowadr[i] + k
            j = mjm.M_colind[adr]
            M[i, j] = M[j, i] = values[adr]
    return M


mjm = mujoco.MjModel.from_xml_string(PENDULUM)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1)
mjw.fwd_position(m, d)                       # kinematics, com_pos, crb -> d.M (and factor_m)

print("nv =", mjm.nv, " nC =", mjm.nC)
print("M_rownnz =", mjm.M_rownnz, " M_rowadr =", mjm.M_rowadr, " M_colind =", mjm.M_colind)
print("d.M shape:", d.M.shape, " values [M00, M10, M11] =", d.M.numpy()[0])
dense = csr_to_dense(mjm, d.M.numpy()[0])
mjd = mujoco.MjData(mjm)
mujoco.mj_forward(mjm, mjd)
full = np.zeros((mjm.nv, mjm.nv))
mujoco.mj_fullM(mjm, mjd, full)
print("dense from d.M:\n", dense)
print("mujoco.mj_fullM:\n", full)
print(f"max |dense - mj_fullM| = {np.abs(dense - full).max():.2e}")
# fwd_position(factorize=True) also ran factor_m: the 2x2 block is stored as a packed upper
# Cholesky factor U (M = U^T U), row-major [U00, U01, U10 = 0, U11], at qLD_block_adr[0] = 0
U = d.qLD.numpy()[0].reshape(2, 2)
print("d.qLD[0] =", d.qLD.numpy()[0], " U^T U =", (U.T @ U).ravel())

HUMANOID_PATH = epath.resource_path("mujoco_warp") / "test_data/humanoid/humanoid.xml"
hum = mujoco.MjModel.from_xml_path(str(HUMANOID_PATH))
pattern = np.zeros((hum.nv, hum.nv), dtype=int)
for i in range(hum.nv):
    for k in range(hum.M_rownnz[i]):
        j = hum.M_colind[hum.M_rowadr[i] + k]
        pattern[i, j] = pattern[j, i] = 1
print(f"HUMANOID: nv = {hum.nv}, nC = {hum.nC} (lower triangle incl. diagonal), "
      f"dense nonzeros = {pattern.sum()} of {hum.nv * hum.nv}")
print("M_rownnz =", hum.M_rownnz.tolist())

if args.dump_json:
    rows = [[int(j) for j in hum.M_colind[hum.M_rowadr[i]:hum.M_rowadr[i] + hum.M_rownnz[i]]]
            for i in range(hum.nv)]
    dof_body = [hum.body(int(b)).name for b in hum.dof_bodyid]
    with open(args.dump_json, "w") as f:
        json.dump({"nv": int(hum.nv), "nC": int(hum.nC), "rows": rows, "dof_body": dof_body,
                   "pendulum": [round(float(v), 4) for v in d.M.numpy()[0]],
                   "pendulum_qLD": [round(float(v), 4) for v in d.qLD.numpy()[0]]},
                  f, separators=(",", ":"))
    print("wrote", args.dump_json)

### Spy plot of the humanoid mass matrix
Dark cells are stored in `d.M` (lower triangle), pale ones are their mirror. Move the `ROW` slider (static fallback without ipywidgets: edit `ROW` and re-run).

In [ ]:
# Interactive spy plot of the HUMANOID mass matrix: CSR lower triangle (as stored in d.M) and its
# mirror. Move the ROW slider (ipywidgets, preinstalled on Colab) to see which columns one CSR row
# stores: the ancestors of that dof. Without ipywidgets the cell draws ROW = 20 once; edit ROW and re-run.
import mujoco
import numpy as np
from etils import epath
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("skipped: matplotlib not installed")

hum = mujoco.MjModel.from_xml_path(
    str(epath.resource_path("mujoco_warp") / "test_data/humanoid/humanoid.xml"))


def spy(ROW=20):
    pattern = np.zeros((hum.nv, hum.nv))
    for i in range(hum.nv):
        cols = hum.M_colind[hum.M_rowadr[i]:hum.M_rowadr[i] + hum.M_rownnz[i]]
        pattern[i, cols] = 2 if i == ROW else 1       # stored (lower triangle)
        pattern[cols, i] = np.maximum(pattern[cols, i], 0.4)  # mirrored (upper, not stored)
    cols = hum.M_colind[hum.M_rowadr[ROW]:hum.M_rowadr[ROW] + hum.M_rownnz[ROW]]
    print(f"row {ROW} ({hum.body(int(hum.dof_bodyid[ROW])).name}): CSR adr {hum.M_rowadr[ROW]}, "
          f"nnz {hum.M_rownnz[ROW]}, columns {cols.tolist()}")
    if plt is not None:
        fig, ax = plt.subplots(figsize=(5.5, 5.5))
        ax.imshow(pattern, cmap="Greens", vmin=0, vmax=2)
        ax.set_title(f"HUMANOID M: nC = {hum.nC} stored of {hum.nv * hum.nv}, row {ROW}")
        ax.set_xlabel("column (dof)")
        ax.set_ylabel("row (dof)")
        plt.show()


try:
    import ipywidgets as widgets
    widgets.interact(spy, ROW=widgets.IntSlider(value=20, min=0, max=hum.nv - 1, description="ROW"))
except ImportError:
    print("static fallback: ipywidgets not installed (try ROW = 6, 15, 20, 26)")
    spy(ROW=20)

## 9. factor_m, solve_m, mul_m and jac

In [ ]:
# Example 11.9 - ex9_solve_mul.py
"""Example 11.9 - factor_m, solve_m and mul_m: x = M^-1 y and back again.

BALL_DROP (shared model M1): one free sphere, mass 1, radius 0.1. Its inertia is
2/5 m r^2 = 0.004 about every axis, and the free joint's 6 dofs are decoupled, so
M = diag(1, 1, 1, 0.004, 0.004, 0.004) and M^-1 * ones = [1, 1, 1, 250, 250, 250].
  mjw.factor_m(m, d)        factor M into d.qLD / d.qLDiagInv
  mjw.solve_m(m, d, x, y)   x = M^-1 y using that stored factor
  mjw.mul_m(m, d, z, x)     z = M x (no factor needed)
Run: python ex9_solve_mul.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8

mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=4, njmax=16)
mjw.fwd_position(m, d, factorize=False)      # build d.M only
mjw.factor_m(m, d)                           # explicit factorization

print("M_rownnz =", mjm.M_rownnz, "-> every row is diagonal-only (a 'simple' free body)")
print("d.M[0] =", d.M.numpy()[0])
print("M_tiles:", [(t.size, t.adr.numpy().tolist()) for t in m.M_tiles],
      " qLD_block_adr =", m.qLD_block_adr.numpy(), " qLD shape:", d.qLD.shape)
print("d.qLDiagInv[0] =", d.qLDiagInv.numpy()[0])

y = wp.ones((d.nworld, mjm.nv), dtype=float)
x = wp.zeros((d.nworld, mjm.nv), dtype=float)
z = wp.zeros((d.nworld, mjm.nv), dtype=float)
mjw.solve_m(m, d, x, y)                      # x = M^-1 y
mjw.mul_m(m, d, z, x)                        # z = M x
print("x = M^-1 ones :", x.numpy()[0])
print("z = M x       :", z.numpy()[0])
print(f"max |z - ones| over {NWORLD} worlds: {np.abs(z.numpy() - 1).max():.2e}")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

In [ ]:
# Example 11.10 - ex10_site_jacobian.py
"""Example 11.10 - mjw.jac: the Jacobian of the pendulum tip, per world, vs mujoco.mj_jacSite.

mjw.jac(m, d, jacp, jacr, point, body) fills jacp/jacr of shape (nworld, 3, nv) for one point per
world (point: wp.array of vec3, body: wp.array of int, both length nworld). It needs subtree_com
and cdof, so run kinematics + com_pos (or fwd_position / forward) first. Here every world has its
own random pose and we ask for the tip site's Jacobian.
Run: python ex10_site_jacobian.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""
ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8

mjm = mujoco.MjModel.from_xml_string(PENDULUM)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
rng = np.random.default_rng(4)
qpos = rng.uniform(-np.pi, np.pi, (NWORLD, mjm.nq))
d.qpos.assign(qpos.astype(np.float32))
mjw.kinematics(m, d)
mjw.com_pos(m, d)

site = mujoco.mj_name2id(mjm, mujoco.mjtObj.mjOBJ_SITE, "tip")
point = wp.array(d.site_xpos.numpy()[:, site], dtype=wp.vec3)          # (nworld,) vec3
body = wp.full(NWORLD, int(mjm.site_bodyid[site]), dtype=int)          # (nworld,) int
jacp = wp.zeros((NWORLD, 3, mjm.nv), dtype=float)
jacr = wp.zeros((NWORLD, 3, mjm.nv), dtype=float)
mjw.jac(m, d, jacp, jacr, point, body)

mjd = mujoco.MjData(mjm)
jp, jr = np.zeros((3, mjm.nv)), np.zeros((3, mjm.nv))
err = 0.0
for w in range(NWORLD):
    mjd.qpos[:] = qpos[w]
    mujoco.mj_kinematics(mjm, mjd)
    mujoco.mj_comPos(mjm, mjd)
    mujoco.mj_jacSite(mjm, mjd, jp, jr, site)
    err = max(err, np.abs(jacp.numpy()[w] - jp).max(), np.abs(jacr.numpy()[w] - jr).max())
print("jacp shape:", jacp.shape, " world 0 q =", qpos[0])
print("world 0 jacp (rows x, y, z; columns hinge1, hinge2):\n", jacp.numpy()[0])
print("world 0 jacr:\n", jacr.numpy()[0])
print(f"max |mjw.jac - mj_jacSite| over {NWORLD} worlds: {err:.2e}")
v_tip = jacp.numpy()[0] @ np.array([1.0, 0.0])
print("tip velocity for qdot = (1, 0):", v_tip, " |v| =", round(float(np.linalg.norm(v_tip)), 4),
      "= distance hinge1 -> tip")

### The stale-factor trap
One factorization per new qpos: compare the four pairings.

In [ ]:
# Example 11.13 - ex13_stale_factor.py
"""Example 11.13 - the stale-factor trap: one factorization per new qpos, never zero.

The factor of M lives in d.qLD / d.qLDiagInv and is only rewritten when someone factors:
  fwd_position(m, d, factorize=True)       (public default)  -> smooth.factor_m
  fwd_acceleration(m, d, factorize=True)   (what _forward passes) -> fused smooth.factor_solve_i
  fwd_acceleration(m, d)                   (public default factorize=False) -> smooth.solve_m,
                                            which only READS the stored factor.
We factor at qpos A (mjw.forward), move every world to a new qpos B and run the stages by hand
with four pairings. The bad mix solves M(B) qacc = f with the factor of M(A): no error is raised,
qacc_smooth is just wrong.
Run: python ex13_stale_factor.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""
ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8

mjm = mujoco.MjModel.from_xml_string(PENDULUM)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)
rng = np.random.default_rng(6)
qpos_a = rng.uniform(-np.pi, np.pi, (NWORLD, mjm.nq)).astype(np.float32)
qpos_b = rng.uniform(-np.pi, np.pi, (NWORLD, mjm.nq)).astype(np.float32)

# reference: a full forward at qpos B
d.qpos.assign(qpos_b)
mjw.forward(m, d)
ref = d.qacc_smooth.numpy().copy()

PAIRINGS = {
    "public defaults: fwd_position() + fwd_acceleration()": (True, False),
    "_forward pairing: fwd_position(False) + fwd_acceleration(True)": (False, True),
    "factor twice: fwd_position() + fwd_acceleration(True)": (True, True),
    "BAD MIX: fwd_position(False) + fwd_acceleration()": (False, False),
}
errors = {}
for name, (fp, fa) in PAIRINGS.items():
    d.qpos.assign(qpos_a)
    mjw.forward(m, d)                           # qLD now holds the factor of M(qpos A)
    d.qpos.assign(qpos_b)                       # move every world
    mjw.fwd_position(m, d, factorize=fp)        # rebuilds d.M = M(B); factors only if fp
    mjw.fwd_velocity(m, d)
    mjw.fwd_actuation(m, d)
    mjw.fwd_acceleration(m, d, factorize=fa)
    err = np.abs(d.qacc_smooth.numpy() - ref).max()
    errors[name] = float(err)
    print(f"{name:66s} max |qacc_smooth - ref| = {err:.2e}")
print("world 0 reference qacc_smooth:", ref[0])
d.qpos.assign(qpos_a); mjw.forward(m, d); d.qpos.assign(qpos_b)
mjw.fwd_position(m, d, factorize=False); mjw.fwd_velocity(m, d)
mjw.fwd_actuation(m, d); mjw.fwd_acceleration(m, d)
print("world 0 bad-mix qacc_smooth  :", d.qacc_smooth.numpy()[0])

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"errors": {k: float(f"{v:.3g}") for k, v in errors.items()},
                   "ref0": np.round(ref[0].astype(float), 4).tolist(),
                   "bad0": np.round(d.qacc_smooth.numpy()[0].astype(float), 4).tolist()}, f)
    print("wrote", args.dump_json)

## 10. Going deeper: the factorization families (`m.M_tiles`, `qLD`)
Every field read here is public on `mjw.Model` (`M_tiles`, `qLD_block_adr`, `is_sparse`).

In [ ]:
# Example 11.12 - ex12_m_tiles.py
"""Example 11.12 (going deeper) - how factor_m routes each block of M: m.M_tiles and qLD.

M is block diagonal: one block per kinematic tree (io.py::_m_blocks uses tree_dofadr/tree_dofnum).
io.py::m_block_layout classifies each block:
  compact   size <= 6 and diagonal-only (nnz == size)   -> reciprocal diagonal, no factor stored
  scalar    size <= 6 and full lower triangle            -> _small_cholesky_factorize_block
  tile      7 .. 64 dofs (M_BLOCK_DENSE_MAX), or a size <= 6 block that is neither compact
            nor a full triangle                          -> _tile_cholesky_factorize_block
  sparse    > 64 dofs                                    -> sparse L'DL (_factor_i_sparse)
put_model turns this into m.M_tiles (TileSet(size, adr, elemid); empty elemid = scalar path),
m.qLD_block_adr (packed offset or the sentinels -2 compact / -1 sparse), m.qLD_block_total and
m.qLD_updates (the sparse level schedule) and m.is_sparse. At the end one fwd_position call on the
HUMANOID measures how much the tile Cholesky factor fills in (threshold printed).
Run: python ex12_m_tiles.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

def box_stack(n=5, size=0.1):
    bodies = "".join(
        f'<body name="box{i}" pos="0 0 {size + 2*size*i + 0.001*i:.4f}"><freejoint/>'
        f'<geom type="box" size="{size} {size} {size}"/></body>' for i in range(n))
    return (f'<mujoco model="box_stack"><option timestep="0.005"/><worldbody>'
            f'<light pos="0 0 3" dir="0 0 -1"/><geom name="floor" type="plane" size="5 5 .1"/>'
            f'{bodies}</worldbody></mujoco>')


def two_arms(n=30):
    """A free base with two hinge chains of n links each: one tree of 6 + 2n dofs."""
    def chain(k, side):
        if k == n:
            return ""
        return (f'<body pos="0 {side * 0.05:.2f} -.05"><joint type="hinge" axis="1 0 0"/>'
                f'<geom type="capsule" fromto="0 0 0 0 0 -.05" size=".01"/>{chain(k + 1, side)}'
                f'</body>')
    return (f'<mujoco model="two_arms"><worldbody><body pos="0 0 3"><freejoint/>'
            f'<geom type="box" size=".1 .1 .1"/>{chain(0, 1)}{chain(0, -1)}</body>'
            f'</worldbody></mujoco>')


HUMANOID_PATH = epath.resource_path("mujoco_warp") / "test_data/humanoid/humanoid.xml"
models = {
    "PENDULUM": mujoco.MjModel.from_xml_string(PENDULUM),
    "HUMANOID": mujoco.MjModel.from_xml_path(str(HUMANOID_PATH)),
    "box_stack(5)": mujoco.MjModel.from_xml_string(box_stack(5)),
    "two_arms(30)": mujoco.MjModel.from_xml_string(two_arms(30)),
}
PATH = {-2: "compact", -1: "sparse"}
dump = {}
for name, mjm in models.items():
    m = mjw.put_model(mjm)
    adr = m.qLD_block_adr.numpy()
    tiles = [(t.size, t.adr.numpy().tolist(), int(t.elemid.size)) for t in m.M_tiles]
    scalar_starts = {a for t in m.M_tiles if t.elemid.size == 0 for a in t.adr.numpy().tolist()}
    blocks = []
    for start, size in zip(mjm.tree_dofadr, mjm.tree_dofnum):
        a = int(adr[start])
        kind = PATH.get(a, "scalar" if int(start) in scalar_starts else "tile")
        blocks.append({"start": int(start), "size": int(size), "kind": kind, "qld": a})
    nlev = [int(u.size) for u in m.qLD_updates]
    print(f"{name}: nv = {mjm.nv}, nC = {mjm.nC}, ntree = {mjm.ntree}, is_sparse = "
          f"{m.is_sparse}")
    print(f"  M_tiles (size, adr, elemid.size) = {tiles}")
    print(f"  blocks = {[(b['start'], b['size'], b['kind']) for b in blocks]}")
    print(f"  qLD_block_total = {m.qLD_block_total}, qLD_updates: {len(nlev)} levels, "
          f"updates per level = {nlev[:8]}{' ...' if len(nlev) > 8 else ''}")
    dump[name] = {"nv": int(mjm.nv), "blocks": blocks, "qld_total": int(m.qLD_block_total),
                  "levels": nlev}

mjm6 = mujoco.MjModel.from_xml_string(box_stack(6))
m6 = mjw.put_model(mjm6)
d6 = mjw.make_data(mjm6, nworld=1, nconmax=48, njmax=192)
print(f"box_stack(6): nv = {mjm6.nv} > 32 -> is_sparse = {m6.is_sparse}, "
      f"d.efc.J.shape = {d6.efc.J.shape}")
mjm5 = models["box_stack(5)"]
m5 = mjw.put_model(mjm5)
d5 = mjw.make_data(mjm5, nworld=1, nconmax=40, njmax=160)
print(f"box_stack(5): nv = {mjm5.nv} <= 32 -> is_sparse = {m5.is_sparse}, "
      f"d.efc.J.shape = {d5.efc.J.shape}")

# Fill-in of the HUMANOID factor at qpos0: d.qLD holds the packed 27 x 27 tile factor U (M = U^T U).
mjh = models["HUMANOID"]
mh = mjw.put_model(mjh)
dh = mjw.make_data(mjh, nworld=1)
mjw.fwd_position(mh, dh)                     # kinematics, crb -> d.M, then factor_m -> d.qLD
nv = mjh.nv
U = dh.qLD.numpy()[0][:nv * nv].reshape(nv, nv)
iu = np.triu_indices(nv)
upper = np.abs(U[iu])
print(f"HUMANOID factor U (upper triangle, {upper.size} slots): "
      f"{int((upper != 0).sum())} nonzero, {int((upper > 1e-6).sum())} with |U_ij| > 1e-6; "
      f"M stores {mjh.nC} (structural)")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump(dump, f, separators=(",", ":"))
    print("wrote", args.dump_json)

## Exercise 11.1 (auto-checked): your own RNE
Compute `c(q, qdot)` for the pendulum with two sweeps over the 2 links and assert it against `d.qfrc_bias` in every world.

In [ ]:
# Exercise 11.1: c(q, qdot) for the PENDULUM with your own two-sweep NumPy RNE.
# Run Example 11.2 first (it defines PENDULUM). Fill in my_rne and make the assert pass.
mjm = mujoco.MjModel.from_xml_string(PENDULUM)
m = mjw.put_model(mjm)
NWORLD = 8
d = mjw.make_data(mjm, nworld=NWORLD)
rng = np.random.default_rng(7)
d.qpos.assign(rng.uniform(-np.pi, np.pi, (NWORLD, 2)).astype(np.float32))
d.qvel.assign(rng.uniform(-3, 3, (NWORLD, 2)).astype(np.float32))
mjw.forward(m, d)


def my_rne(q, qd, xanchor, xipos):
    """Return c = [tau1, tau2] for qacc = 0 (Coriolis + centrifugal + gravity)."""
    ...  # forward sweep: w1, w2, a_c1, a_c2 (gravity trick: base accelerates up by g)
    ...  # backward sweep: forces F2, F1 and the torques about each hinge axis


q, qd = d.qpos.numpy(), d.qvel.numpy()
xa, xi = d.xanchor.numpy(), d.xipos.numpy()
if my_rne(q[0], qd[0], xa[0], xi[0]) is None:
    print("my_rne returns None: fill in the two sweeps, then re-run this cell (or open the solution).")
else:
    mine = np.array([my_rne(q[w], qd[w], xa[w], xi[w]) for w in range(NWORLD)])
    assert np.abs(mine - d.qfrc_bias.numpy()).max() < 1e-4
    print("Exercise 11.1 passed")

In [ ]:
#@title Exercise 11.1: solution (double-click to show the code)
mjm = mujoco.MjModel.from_xml_string(PENDULUM)
m = mjw.put_model(mjm)
NWORLD = 8
d = mjw.make_data(mjm, nworld=NWORLD)
rng = np.random.default_rng(7)
d.qpos.assign(rng.uniform(-np.pi, np.pi, (NWORLD, 2)).astype(np.float32))
d.qvel.assign(rng.uniform(-3, 3, (NWORLD, 2)).astype(np.float32))
mjw.forward(m, d)


def my_rne(q, qd, xanchor, xipos):
    """c(q, qdot): RNE with qacc = 0 and the gravity trick (base accelerates up by g)."""
    e = np.array([0.0, 1.0, 0.0])                     # both hinge axes (world y)
    g = -mjm.opt.gravity[2]
    mass = mjm.body_mass[1:3]
    p1, p2 = xanchor[0], xanchor[1]
    c1, c2 = xipos[1], xipos[2]
    # forward sweep (root -> leaf): angular velocity and linear acceleration
    w1 = e * qd[0]
    w2 = w1 + e * qd[1]
    a_p1 = np.array([0.0, 0.0, g])                    # gravity trick
    a_c1 = a_p1 + np.cross(w1, np.cross(w1, c1 - p1))
    a_p2 = a_p1 + np.cross(w1, np.cross(w1, p2 - p1))
    a_c2 = a_p2 + np.cross(w2, np.cross(w2, c2 - p2))
    # backward sweep (leaf -> root): force and moment carried to each joint
    F2 = mass[1] * a_c2                               # alpha = 0 and w x I w = 0 (planar)
    F1 = mass[0] * a_c1
    tau2 = e @ np.cross(c2 - p2, F2)
    tau1 = e @ (np.cross(c1 - p1, F1) + np.cross(c2 - p1, F2))
    return np.array([tau1, tau2])


q, qd = d.qpos.numpy(), d.qvel.numpy()
xa, xi = d.xanchor.numpy(), d.xipos.numpy()
mine = np.array([my_rne(q[w], qd[w], xa[w], xi[w]) for w in range(NWORLD)])
err = np.abs(mine - d.qfrc_bias.numpy()).max()
print("world 0 mine:", mine[0], " qfrc_bias:", d.qfrc_bias.numpy()[0], f" max err {err:.2e}")
assert err < 1e-4
print("Exercise 11.1 passed")

## Exercise 11.2 (auto-checked): predict nbranch
A free hub with three 2-link arms.

In [ ]:
# Exercise 11.2: a hub on a free joint with 3 arms of 2 links each. Predict before you run.
def arm(k):
    return (f'<body pos="{np.cos(k * 2.094):.3f} {np.sin(k * 2.094):.3f} 0">'
            f'<joint type="hinge" axis="0 0 1"/><geom type="sphere" size=".05"/>'
            f'<body pos=".3 0 0"><joint type="hinge" axis="0 1 0"/><geom type="sphere" size=".05"/>'
            f'</body></body>')


STAR = ('<mujoco><worldbody><body name="hub" pos="0 0 1"><freejoint/>'
        '<geom type="cylinder" size=".2 .05"/>' + "".join(arm(k) for k in range(3)) +
        '</body></worldbody></mujoco>')
mjm = mujoco.MjModel.from_xml_string(STAR)
m = mjw.put_model(mjm)

nbranch_pred = ...    # your prediction
nvisit_pred = ...     # total body visits of the branch walk = len(m.body_branches)
if nbranch_pred is ... or nvisit_pred is ...:
    print("Replace the two ... with your predictions, then re-run this cell (or open the solution).")
else:
    assert m.nbranch == nbranch_pred, m.nbranch
    assert len(m.body_branches.numpy()) == nvisit_pred
    print("Exercise 11.2 passed")

In [ ]:
#@title Exercise 11.2: solution (double-click to show the code)
# Exercise 11.2 solution (self-contained: rebuilds the STAR model).
def arm(k):
    return (f'<body pos="{np.cos(k * 2.094):.3f} {np.sin(k * 2.094):.3f} 0">'
            f'<joint type="hinge" axis="0 0 1"/><geom type="sphere" size=".05"/>'
            f'<body pos=".3 0 0"><joint type="hinge" axis="0 1 0"/><geom type="sphere" size=".05"/>'
            f'</body></body>')


STAR = ('<mujoco><worldbody><body name="hub" pos="0 0 1"><freejoint/>'
        '<geom type="cylinder" size=".2 .05"/>' + "".join(arm(k) for k in range(3)) +
        '</body></worldbody></mujoco>')
mjm = mujoco.MjModel.from_xml_string(STAR)
m = mjw.put_model(mjm)

nbranch_pred = 3          # one branch per leaf body: the three outer links
nvisit_pred = 3 * 3       # each branch visits hub + inner link + outer link
assert m.nbranch == nbranch_pred, m.nbranch
assert len(m.body_branches.numpy()) == nvisit_pred
print("nbranch =", m.nbranch, " body_branches =", m.body_branches.numpy().tolist())
print("body_subtreenum =", m.body_subtreenum.numpy().tolist())
print("Exercise 11.2 passed")

## Stretch: predict `M_tiles` for 3 free bodies + a 7-dof arm
Write your prediction (sizes, start dofs, which path, `qLD_block_total`) before running the cell.

In [ ]:
# Stretch: check your prediction of M_tiles for 3 free spheres + a 7-hinge arm.
def scene(offset):
    frees = "".join(f'<body pos="{i} 0 1"><freejoint/>'
                    f'<geom type="sphere" size=".1" pos="{offset} 0 0"/></body>' for i in range(3))
    links, close = "", ""
    for k in range(7):
        ax = "0 0 1" if k % 2 == 0 else "0 1 0"
        links += (f'<body pos="0 0 .1"><joint type="hinge" axis="{ax}"/>'
                  f'<geom type="capsule" fromto="0 0 0 0 0 .1" size=".02"/>')
        close += "</body>"
    return f'<mujoco><worldbody>{frees}<body pos="0 2 0">{links}{close}</body></worldbody></mujoco>'


for offset in (0.0, 0.05):      # sphere at the body origin, then off-centre
    mjm = mujoco.MjModel.from_xml_string(scene(offset))
    m = mjw.put_model(mjm)
    print(f"offset {offset}: M_tiles =",
          [(t.size, t.adr.numpy().tolist(), t.elemid.size) for t in m.M_tiles],
          "qLD_block_total =", m.qLD_block_total)
    print("   qLD_block_adr =", m.qLD_block_adr.numpy().tolist())